In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_dir = "../../data/"

target = pd.read_csv(data_dir + "train.csv")

book = pd.read_parquet(data_dir + "book_train.parquet")
trade = pd.read_parquet(data_dir + "trade_train.parquet")

In [ ]:

# 两档总挂单量。注意：算 WAP1 不用这个，用的是一档的买卖量之和
book["tot_size"] = book["bid_size1"] + book["bid_size2"] + book["ask_size1"] + book["ask_size2"]

# WAP（加权中间价）：买价乘卖量，卖价乘买量，权重是交叉的
# 卖单多，说明价格更可能往下走，所以 WAP 更靠近买价；买单多则更靠近卖价
book["wap1"] = ((book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"])
             / (book["bid_size1"] + book["ask_size1"])).astype("float32")

book["wap2"] = ((book["bid_price2"] * book["ask_size2"] + book["ask_price2"] * book["bid_size2"])
             / (book["bid_size2"] + book["ask_size2"])).astype("float32")

book["wap12"] = ((book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"] + 
            book["bid_price2"] * book["ask_size2"] + book["ask_price2"] * book["bid_size2"])
             / book["tot_size"]).astype("float32")

book["log_wap1"] = np.log(book["wap1"])
book["log_ret1"] = book.groupby(by = ["stock_id", "time_id"], observed=True)["log_wap1"].diff()

book["log_wap2"] = np.log(book["wap2"])
book["log_ret2"] = book.groupby(by = ["stock_id", "time_id"], observed=True)["log_wap2"].diff()

book["log_wap12"] = np.log(book["wap12"])
book["log_ret12"] = book.groupby(by = ["stock_id", "time_id"], observed=True)["log_wap12"].diff()

In [ ]:
def realized_vol(series):
    return np.sqrt(np.sum(np.square(series)))

rv = book.groupby(by = ["stock_id", "time_id"], observed=True)[["log_ret1", "log_ret2", "log_ret12"]].agg(realized_vol).reset_index()

rv.rename(columns={"log_ret1":"rv1", "log_ret2":"rv2", "log_ret12":"rv12"}, inplace=True)

rv

,stock_id,time_id,rv1,rv1,rv12
0,0,5,0.006999,0.006999,0.006999
1,0,11,0.002476,0.002476,0.002476
2,0,16,0.004801,0.004801,0.004801
3,0,31,0.003637,0.003637,0.003637
4,0,62,0.003257,0.003257,0.003257
...,...,...,...,...,...
428927,99,32751,0.001835,0.001835,0.001835
428928,99,32753,0.001960,0.001960,0.001960
428929,99,32758,0.001981,0.001981,0.001981
428930,99,32763,0.003033,0.003033,0.003033


In [ ]:
combined = pd.merge(target, rv, how = "outer", on = ["stock_id", "time_id"])
combined.isna().any()


stock_id    False
time_id     False
target      False
rv1         False
rv1         False
rv12        False
dtype: bool

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(np.log(combined["rv1"]), np.log(combined["target"]), s=1, alpha=0.3)
plt.show()

,stock_id,time_id,target,rv1,rv1,rv12
0,0,5,0.004136,0.006999,0.006999,0.006999
1,0,11,0.001445,0.002476,0.002476,0.002476
2,0,16,0.002168,0.004801,0.004801,0.004801
3,0,31,0.002195,0.003637,0.003637,0.003637
4,0,62,0.001747,0.003257,0.003257,0.003257
...,...,...,...,...,...,...
428927,126,32751,0.003461,0.005877,0.005877,0.005877
428928,126,32753,0.003113,0.004991,0.004991,0.004991
428929,126,32758,0.004070,0.006019,0.006019,0.006019
428930,126,32763,0.003357,0.005362,0.005362,0.005362


In [10]:
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2))

# baseline：直接用当前窗口的 RV 预测 target
rmspe(combined["target"], combined["rv1"])


ValueError: cannot reindex on an axis with duplicate labels